# 09 · Memory and Persistence: What Survives

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama, and the embedding model
> (`ollama pull nomic-embed-text`) for §2–3.

Two different questions are often mixed up:

- **Memory**: what should agents *remember* across runs? ("Albers Markt wants
  reports on Mondays.")
- **Persistence**: if the process dies halfway through a workflow, what is
  *saved*, and how do we continue?

CrewAI has an answer for each: a unified **Memory** for the first, and **Flow
persistence** plus **crew checkpoints** for the second. This notebook tests each
one, including with **real killed processes**, and reports exactly what held up.

**You will learn**
- CrewAI's unified `Memory`: remember, recall, and what saving costs
- Giving a crew memory, so a fact from one run is used in the next
- Flow persistence with `@persist`: state survives a kill, but the flow **restarts from the top**
- Crew checkpoints (`CheckpointConfig`): what worked and what didn't in our tests
- Which mechanism to use for which job

**Prerequisites:** notebooks 02, 04 and 06.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import shutil, subprocess, sys, textwrap, pathlib, glob, os
from crewai import Agent, Task, Crew
from crewai.memory import Memory
from common import embedder_config

llm = get_llm()
WORK = pathlib.Path('checkpoints')
WORK.mkdir(exist_ok=True)

## 1 · Why: memory vs persistence

| | Memory | Persistence |
|---|---|---|
| Question | what should agents *know* next time? | where did this run get to, and what did it have? |
| Content | facts, preferences, lessons | the workflow's state and position |
| Lifetime | across many runs and conversations | one run (until it finishes) |
| CrewAI feature | `Memory`, `Crew(memory=...)` | `@persist` on Flows, `CheckpointConfig` on crews |

## 2 · Unified memory: remember and recall

`Memory` is a store of facts with **semantic recall**. It embeds each memory with
an embedding model and finds relevant ones by meaning, not keywords. When you save,
an **LLM analyses** the memory (importance, category, **scope**), so saving costs a
model call, and runs in the background. Storage is local (LanceDB) in a folder you choose.

In [ ]:
STORE = WORK / 'memory_store'
shutil.rmtree(STORE, ignore_errors=True)

memory = Memory(llm=llm, embedder=embedder_config(), storage=str(STORE))
memory.remember('Ada prefers answers as short bullet points.')
memory.remember('The Utrecht permit forbids robots on the Oudegracht wharf between 11:00 and 15:00.')
memory.remember('Albers Markt is the customer for the Q3 rollout of 180 robots.')
memory.drain_writes()                      # saving runs in the background; wait for it to finish

print('What the LLM filed, and where:')
for r in memory.list_records():
    print(f'  scope={r.scope:<34} categories={r.categories}\n      "{r.content}"')

Look at how each memory was **filed**. On save, an LLM chose a *scope* (a
folder-like path) and *categories* for each fact. Check whether it filed them
sensibly. In our runs it put "Albers Markt is the customer…" under a
`/regulations` scope with regulation categories, which is plainly wrong.

Now recall. `recall()` has two depths:

- `depth='deep'` (the default): an LLM-guided search that analyses the query and
  explores the likely scopes.
- `depth='shallow'`: plain vector similarity over the stored memories.

In [ ]:
for question in ('How should I format answers for Ada?', 'When can robots use the wharf paths?'):
    print(question)
    for depth in ('deep', 'shallow'):
        hits = memory.recall(question, depth=depth)
        print(f'  {depth:<7} → {hits[0].record.content if hits else "(nothing)"}')

The wharf question works both ways: "wharf paths" finds "Oudegracht wharf …
11:00 and 15:00" by meaning, not keywords. That's semantic recall through
embeddings.

The Ada question is the lesson. In our runs the **deep** recall didn't return the
Ada memory at all (it searched the scopes it guessed, and the Ada fact was
filed elsewhere), while **shallow** vector search ranked it first. The "smart"
path added two LLM judgements (filing and searching), and each one can be wrong.
Check what was stored (`list_records()`), and test recall on questions you
know the answer to before trusting it.

## 3 · A crew with memory

Pass a `Memory` to a crew and its agents use it automatically: relevant memories
are recalled into the prompt before a task, and new facts are extracted and saved
after it. Two **separate runs**, two **new crews**:

In [ ]:
shutil.rmtree(STORE, ignore_errors=True)
shared_memory = Memory(llm=llm, embedder=embedder_config(), storage=str(STORE))

def assistant_crew():
    a = Agent(role='Account assistant', goal='Help the Albers Markt account team', backstory='Helpful.', llm=get_llm())
    return Crew(agents=[a], tasks=[Task(description='{message}', expected_output='A short reply', agent=a)],
                memory=shared_memory)

first = await assistant_crew().kickoff_async(
    inputs={'message': 'Note for the future: Albers Markt wants weekly status reports every Monday.'})
print('run 1:', first.raw)
print('saved :', [getattr(r, 'content', str(r)) for r in shared_memory.list_records()])

second = await assistant_crew().kickoff_async(inputs={'message': 'How often does Albers Markt want status reports?'})
print('\nrun 2:', second.raw)

Run 2 was a new crew with a new agent. It knew the answer only because the
memory store did. Note that the extracted memories are the **LLM's rewording** of
what was said, so what gets remembered (and how) is itself a model decision.
Inspect `list_records()` to see what your agents actually stored.

Memory raises real design questions: stale facts, contradictions, personal data,
and which runs should share which memories (`memory.scope(...)` gives separate
namespaces).

## 4 · Flow persistence: surviving a crash

`@persist(...)` on a Flow class saves the flow's **state after every method**,
keyed by the state's `id`. Kicking off again with `inputs={'id': <that id>}` restores it.

We'll test that with a real kill. The script's third step calls `os._exit(1)`,
like `kill -9`. Each step logs to `flow.log`, so we can see what re-ran:

In [ ]:
(WORK / 'research_flow.py').write_text(textwrap.dedent(f"""
    import os, sys, sqlite3
    sys.path.insert(0, {str(SECTION)!r})
    import common
    from pydantic import BaseModel
    from crewai.flow import Flow, start, listen, persist
    from crewai.flow.persistence import SQLiteFlowPersistence

    class State(BaseModel):
        notes: list[str] = []
        report: str = ''

    def log(step):
        open('flow.log', 'a').write(step + ' ')

    @persist(SQLiteFlowPersistence('flow_state.db'))
    class ResearchFlow(Flow[State]):
        @start()
        def fetch(self):
            if 'battery notes' in self.state.notes:        # idempotency guard
                log('fetch(skipped)'); return
            log('fetch'); self.state.notes.append('battery notes')

        @listen(fetch)
        def analyse(self):
            if 'analysis' in self.state.notes:
                log('analyse(skipped)'); return
            log('analyse'); self.state.notes.append('analysis')

        @listen(analyse)
        def write(self):
            log('write')
            if os.environ.get('CRASH'):
                os._exit(1)                                  # the process dies here
            self.state.report = ' + '.join(self.state.notes)
            return self.state.report

    if sys.argv[1] == 'start':
        ResearchFlow(suppress_flow_events=True).kickoff()
    else:
        flow_id = sqlite3.connect('flow_state.db').execute(
            'select flow_uuid from flow_states order by rowid desc limit 1').fetchone()[0]
        flow = ResearchFlow(suppress_flow_events=True)
        print('RESTORED', flow.kickoff(inputs={{'id': flow_id}}), '| state:', flow.state.notes)
"""))

def run(mode, **env):
    out = subprocess.run([sys.executable, 'research_flow.py', mode], cwd=WORK, capture_output=True, text=True,
                         env={**os.environ, **env})
    print(f'$ python research_flow.py {mode}   exit={out.returncode}  ' +
          ' '.join(l for l in out.stdout.splitlines() if l.startswith('RESTORED')))

for f in ('flow_state.db', 'flow.log'):
    (WORK / f).unlink(missing_ok=True)
run('start', CRASH='1')
import sqlite3
print('saved rows:', sqlite3.connect(WORK / 'flow_state.db').execute('select method_name from flow_states').fetchall())
run('resume')
print('\nsteps executed across both processes:', (WORK / 'flow.log').read_text())

Two important observations:

1. **The state survived.** After the kill, the database held the state saved after
   `fetch` and after `analyse`, and the new process restored both notes.
2. **The flow restarted from `@start`.** `@persist` restores *state*, not
   *position*: `fetch` and `analyse` ran again, and only our **idempotency guards**
   ("already have the notes? skip") stopped them from repeating their work.

So with Flow persistence, **every step must check the state before doing work**.
Without the guards, a restored flow would fetch and analyse again. That's
different from LangGraph, which resumes at the next step, and it's the most
important thing to know about `@persist`.

## 5 · Crew checkpoints: `CheckpointConfig`

Crews (and flows and agents) accept `checkpoint=CheckpointConfig(...)`, which
writes a checkpoint when chosen events fire (e.g. `task_completed`), and
`kickoff(from_checkpoint=...)` to restore. It's a newer feature, so we tested it
the same way. First, a normal run with a **class-based** tool:

In [ ]:
from crewai import CheckpointConfig
from crewai.tools import BaseTool, tool

class RecordStep(BaseTool):
    name: str = 'Record step'
    description: str = 'Record that a step was completed.'
    def _run(self, step: str) -> str:
        return f'recorded {step}'

def checkpointed_crew(tools, location):
    a = Agent(role='Clerk', goal='Record steps', backstory='Always uses the tool.', llm=get_llm(), tools=tools)
    tasks = [Task(description=f'Call the Record step tool with step="{n}", then reply done.',
                  expected_output='done', agent=a) for n in ('one', 'two')]
    return Crew(agents=[a], tasks=tasks,
                checkpoint=CheckpointConfig(location=location, on_events=['task_completed']))

loc = WORK / 'crew_ckpt_class_tool'
shutil.rmtree(loc, ignore_errors=True)
await checkpointed_crew([RecordStep()], str(loc)).kickoff_async()
print('checkpoint files (class tool):', len([f for f in glob.glob(f'{loc}/**/*.json', recursive=True)]))

Checkpoints were written. Now the same crew with a **function** tool (`@tool`):

In [ ]:
@tool('Record step')
def record_step(step: str) -> str:
    """Record that a step was completed."""
    return f'recorded {step}'

import logging
logging.getLogger('crewai.state.checkpoint_listener').setLevel(logging.ERROR)   # its warning is very long
loc = WORK / 'crew_ckpt_function_tool'
shutil.rmtree(loc, ignore_errors=True)
await checkpointed_crew([record_step], str(loc)).kickoff_async()
print('checkpoint files (function tool):', len([f for f in glob.glob(f'{loc}/**/*.json', recursive=True)]))

**Zero files, and the run "succeeded".** The checkpoint listener couldn't
serialise the crew's state, because a function tool can't be turned into JSON.
It logged a warning and carried on. In a real system you'd believe you had
checkpoints, and you wouldn't.

We then tested recovery the way it matters: a crew whose third task **kills the
process**, after two tasks had completed. In our runs (CrewAI 1.15, class-based
tool, both a hard kill and a `SystemExit`), **no checkpoint files existed
afterwards**, so there was nothing to restore. Checkpoint writes are triggered
from CrewAI's event system, and in our tests they didn't reach the disk before
the process ended. Normal runs, as above, did write them.

The lesson isn't "never use it". This is a new, evolving feature. The lesson
is: **test your recovery path by actually killing the process**, the way these
notebooks do, before relying on any checkpoint mechanism.

## 6 · Which mechanism for which job

| You need… | Use | Caveat from our tests |
|---|---|---|
| agents to remember facts across runs | `Memory` / `Crew(memory=...)` | saving costs an LLM call; memories are LLM-reworded and LLM-filed; deep recall missed a fact that shallow recall found |
| a multi-step workflow to survive a crash | a **Flow** with `@persist` | restores state, **restarts from `@start`**: make steps idempotent |
| a paused human approval | Flow + `@human_feedback` + persistence (notebook 08) | worked reliably across processes |
| checkpoint/restore of a crew mid-run | `CheckpointConfig` | function tools break it silently; no checkpoint survived our crash tests |

For the project, we'll use a Flow with `@persist` and idempotent steps: the
option that held up under a real kill.

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Treating memory as persistence (or vice versa) | the wrong thing survives | memory = knowledge; persistence = workflow state |
| Non-idempotent flow steps with `@persist` | work repeats after a restore | check state before doing work |
| Function tools in checkpointed crews | no checkpoints, only a warning | `BaseTool` classes, and check the files exist |
| Trusting checkpoints you've never restored | surprises in production | kill the process in a test and restore |
| Unscoped shared memory | facts leak between users/teams | `memory.scope(...)`, separate stores |
| Trusting deep recall blindly | a stored fact isn't recalled | inspect `list_records()`; compare `depth='shallow'`; test known questions |

## 8 · Check yourself

<details><summary><b>1.</b> What's the difference between memory and persistence?</summary>

Memory is knowledge agents should carry across runs (facts, preferences). Persistence saves a workflow's state so a run can continue after a crash or pause.
</details>

<details><summary><b>2.</b> After a crash and restore, which steps of a <code>@persist</code> Flow run again?</summary>

All of them, from `@start`: persistence restores the state, not the position. Steps must check the state and skip work that's already done (idempotency).
</details>

<details><summary><b>3.</b> A crew with <code>@tool</code> function tools and a <code>CheckpointConfig</code> runs successfully. Are checkpoints being written?</summary>

In our tests, no. Serialising the function tool failed, and the listener only logged a warning. Use `BaseTool` classes and verify that the files exist.
</details>

<details><summary><b>4.</b> Why does saving to <code>Memory</code> cost a model call?</summary>

The unified memory uses an LLM to analyse each memory (importance, category, scope) and to extract facts, in addition to embedding it for search.
</details>

## Takeaway

Use **Memory** for knowledge that should outlive a run: it recalls by meaning,
but saving costs model calls, the stored text is the LLM's paraphrase and filing,
and deep recall can miss what shallow vector search finds, so test it. For
workflows that must survive a crash, a **Flow with `@persist`** held up under a
real kill, as long as every step is idempotent, because the flow restarts from
the top. Crew checkpoints didn't survive our crash tests, so always test
recovery by actually killing the process.

Next → `10_observability.ipynb`: seeing what a crew is doing.